# Asistente LLM · TFM Energía UCM

Registro de todo el código del asistente en un solo sitio, para el anexo. No es un notebook de
entrenamiento: el asistente es un servicio vivo (FastAPI + tool use de Claude), así que aquí no
se "entrena" nada -- se documenta la arquitectura real y se extrae el código fuente en vivo con
`inspect.getsource()`, directo de los módulos. Así el anexo nunca queda desincronizado del
código real: si `herramientas.py` cambia, esta celda muestra el cambio la próxima vez que se
ejecute, sin copiar y pegar a mano.

## Arquitectura, en una frase

Claude (tool use) nunca inventa números: entiende la pregunta, elige qué función determinista de
`herramientas.py` llamar y con qué parámetros, y redacta la respuesta a partir de lo que esa
función devuelve. Tres piezas:

- **`herramientas.py`** -- funciones deterministas contra Postgres (precios, simulaciones,
  capacidad instalada). Devuelven datos, nunca texto redactado.
- **`chat.py`** -- envuelve cada función como `@beta_tool`, arma el `system prompt`, orquesta el
  bucle de `tool_runner`.
- **`indexar_documentacion.py`** -- RAG aparte (no tool use): embeddings de `docs/notas_memoria_tfm.md`
  y docstrings del código, en `documentacion_embeddings` (pgvector).

Regla central del prompt: "predicción" solo existe para D+1 (`prediccion_d_mas_1`). Cualquier
otro horizonte se responde como referencia histórica o escenario, nunca disfrazado de predicción
del modelo (nota 33).

In [1]:
import sys, inspect
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modelos" / "asistente").is_dir())
sys.path.append(str(REPO / "modelos" / "asistente"))
sys.path.append(str(REPO / "ingesta"))

import chat
import herramientas
import indexar_documentacion

print(f"herramientas registradas: {len(chat.TOOLS)}")
print(f"modelo por defecto: {chat.MODELO_POR_DEFECTO}")

herramientas registradas: 13
modelo por defecto: claude-opus-5


## 1 · El prompt del sistema

El texto exacto que fija las reglas (predicción vs. referencia vs. escenario, formato de
tablas/gráficas, cuándo usar SQL de solo lectura como último recurso).

In [2]:
print(chat.SYSTEM_PROMPT)

Eres el asistente del proyecto de prediccion de precio electrico y baterias (TFM UCM).

ALCANCE: no tienes acceso a internet ni busqueda web -- todo lo que sabes de datos viene EXCLUSIVAMENTE
de tus herramientas (que consultan la base de datos y documentacion de este proyecto). Si te preguntan
algo que no tiene que ver con el proyecto (cultura general, otras noticias, otros mercados, cualquier
tema ajeno), NO respondas la pregunta aunque la sepas de tu entrenamiento -- di brevemente que estas
limitado al alcance de este proyecto y ofrece en que si puedes ayudar. Nunca dejes la impresion de que
"sabes de todo": tu utilidad esta en ser fiable dentro de un alcance concreto, no en parecer generalista.

REGLA MAS IMPORTANTE, NUNCA LA ROMPAS: solo existe una "prediccion" real -- la del dia siguiente
(D+1), que sale de `prediccion_d_mas_1`. Para cualquier otro horizonte (una semana, un mes, un
año, un rango de años futuro) NUNCA respondas como si el modelo lo hubiera predicho -- usa
`precio_h

## 2 · Las herramientas (`herramientas.py`)

Cada una es determinista: mismo input, mismo output, sin pasar por el LLM. `chat.py` solo las
envuelve con `@beta_tool`; el código real vive aquí.

In [3]:
FUNCIONES_PUBLICAS = [
    "precio_historico_percentiles", "precio_historico_serie", "precio_tabla_horaria",
    "precio_tendencia_mensual", "precio_negativos", "precio_horas_negativas",
    "prediccion_d_mas_1", "simular_bateria", "simular_autoconsumo_solar",
    "precio_futuro_curva", "extrapolar_consumo_cliente", "buscar_documentacion",
    "capacidad_instalada", "consulta_sql_lectura",
]
for nombre in FUNCIONES_PUBLICAS:
    fn = getattr(herramientas, nombre)
    print(f"{'=' * 76}\n{nombre}\n{'=' * 76}")
    print(inspect.getsource(fn))
    print()

precio_historico_percentiles
def precio_historico_percentiles(hora: int | None = None, mes: int | None = None,
                                  dia_semana: int | None = None,
                                  anio_desde: int | None = None, anio_hasta: int | None = None) -> dict:
    """Percentiles del precio REAL historico, filtrado por hora/mes/dia de la semana.

    Es la herramienta de "extrapolacion" honesta: no predice nada, describe como se ha
    comportado el precio en circunstancias parecidas en el pasado. `dia_semana`: 0=lunes,
    6=domingo (convencion de pandas .dayofweek). Si no se filtra nada, describe todo el
    historico disponible.

    Devuelve un dict con n_horas, media, p10, p25, p50 (mediana), p75, p90, min, max -- y un
    campo `etiqueta` que dice explicitamente que esto es referencia historica, no prediccion,
    para que el LLM lo traslade asi a la respuesta.
    """
    conn = _conectar()
    try:
        df = pd.read_sql("SELECT datetime, es_esios FROM spot

## 3 · El orquestador (`chat.py`)

Cómo se envuelven las herramientas y se llama al modelo. `preguntar` es la versión de terminal;
`preguntar_con_imagenes` es la que usa producción (también recoge las gráficas que el modelo
genere con `code_execution`).

In [4]:
for nombre in ["preguntar", "preguntar_con_imagenes", "_registrar_historial"]:
    print(f"{'=' * 76}\n{nombre}\n{'=' * 76}")
    print(inspect.getsource(getattr(chat, nombre)))
    print()

preguntar
def preguntar(pregunta: str, modelo: str = MODELO_POR_DEFECTO) -> str:
    """Hace una pregunta al asistente y devuelve su respuesta final en texto (sin graficas --
    para eso, `preguntar_con_imagenes`). Se mantiene igual que antes para no romper el uso ya
    existente en terminal."""
    client = anthropic.Anthropic(api_key=load_anthropic_key())

    runner = client.beta.messages.tool_runner(
        model=modelo,
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        tools=TOOLS,
        messages=[{"role": "user", "content": pregunta}],
    )

    ultimo = None
    tokens_entrada = tokens_salida = 0
    for mensaje in runner:
        ultimo = mensaje
        tokens_entrada += mensaje.usage.input_tokens
        tokens_salida += mensaje.usage.output_tokens
    if ultimo is None:
        _registrar_historial(pregunta, "(el asistente no devolvio respuesta)", modelo, tokens_entrada, tokens_salida)
        return "(el asistente no devolvio respuesta)"
    texto = next(

## 4 · RAG documental (`indexar_documentacion.py`)

Aparte del tool use: embeddings locales (fastembed, sin llamar a ninguna API para esto) de
`docs/notas_memoria_tfm.md` y docstrings del código, guardados en pgvector. `buscar_documentacion`
(sección 2) es la herramienta que consulta esto desde el chat.

In [5]:
print(inspect.getsource(indexar_documentacion))

r"""
TFM Energia UCM - Indexar la documentacion para busqueda semantica (RAG documental) (30-ago-2026)

Esto SI es RAG clasico: trocea `docs/notas_memoria_tfm.md` y `docs/columnas_pendientes_equipo.md`
en unidades semanticas (cada nota numerada "## N. Titulo" ya es un chunk coherente -- no hace
falta partir por tamaño de texto), genera un embedding por chunk con un modelo LOCAL (sin clave de
API adicional: `fastembed`, modelo multilingue `paraphrase-multilingual-MiniLM-L12-v2`, 384
dimensiones) y los guarda en Postgres con `pgvector` (extension ya instalada, verificado
29-ago-2026).

Por que embeddings locales y no una API: la clave de Anthropic no cubre embeddings (Anthropic
recomienda Voyage AI, una cuenta aparte) -- para no sumar una segunda dependencia de pago a un
prototipo, se usa un modelo pequeño que corre en la propia maquina. Si mas adelante se quiere
mejor calidad de busqueda, cambiar a Voyage es sustituir esta funcion de embedding, no rehacer el
diseño.

FUENTE DE LA DOCUME

## 5 · El endpoint (`production/api/main.py`)

`main.py` monta media docena de routers y ficheros estáticos -- importarlo entero aquí arrastraría
todo eso sin necesidad. Se extrae solo lo relevante por AST, directo del fichero real, mismo
principio que el resto del notebook.

In [6]:
import ast

fuente_main = (REPO / "production" / "api" / "main.py").read_text(encoding="utf-8")
arbol = ast.parse(fuente_main)
lineas = fuente_main.splitlines()

for nodo in ast.walk(arbol):
    nombre = getattr(nodo, "name", None)
    if nombre in ("PreguntaAsistente", "asistente"):
        trozo = "\n".join(lineas[nodo.lineno - 1:nodo.end_lineno])
        print(f"{'=' * 76}\n{nombre}\n{'=' * 76}")
        print(trozo)
        print()

PreguntaAsistente
class PreguntaAsistente(BaseModel):
    pregunta: str
    # Opcional: para comparar coste/calidad entre modelos desde la propia pagina sin tocar
    # codigo. Si se omite, chat.py usa MODELO_POR_DEFECTO (claude-opus-5).
    modelo: str | None = None

asistente
def asistente(cuerpo: PreguntaAsistente):
    """Reenvia la pregunta al asistente (LLM + herramientas, ver modelos/asistente/chat.py).

    Requiere `anthropic_api_key` en el credentials.json de la maquina donde corre esto -- cada
    persona usa su propia clave local, no una compartida en el servidor (ver nota 33/decision de
    seguridad: es una clave con creditos reales, a diferencia del resto de credenciales del
    proyecto). Si no esta configurada, se devuelve un error claro en vez de que la pagina falle
    en silencio.
    """
    from chat import preguntar_con_imagenes, MODELO_POR_DEFECTO
    if cuerpo.modelo and cuerpo.modelo not in MODELOS_PERMITIDOS:
        raise HTTPException(400, f"Modelo '{cuerpo

## 6 · Demo en vivo (opcional)

Requiere `anthropic_api_key` en `credentials.json` y consume tokens reales. `EJECUTAR = False`
por defecto a propósito.

In [7]:
EJECUTAR = False

if not EJECUTAR:
    print("EJECUTAR = False. Cambia a True en esta celda para probarlo de verdad.")
else:
    pregunta = "¿Cuántas horas de precio negativo ha habido este año?"
    print(f"Pregunta: {pregunta}\n")
    print(chat.preguntar(pregunta))

EJECUTAR = False. Cambia a True en esta celda para probarlo de verdad.
